# 01 Synthetic Data

This notebook introduces the synthetic dataset that drives the local DSP demo.

The dataset has to support four things at once:
- a user profile schema that can live in Redis,
- a campaign schema with both hard filters and soft ranking weights,
- coarse segments for Redis Set-based candidate generation,
- labels generated from a transparent truth model so offline evaluation is meaningful.

The walkthrough below mirrors the actual code used by the service.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise RuntimeError('Could not locate repo root')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.models import Campaign, UserProfile
from data.common import campaign_is_eligible, click_label, click_probability, read_jsonl, segment_for, truth_score
from data.synthetic import ensure_synthetic_dataset

DATASET_DIR = REPO_ROOT / 'data' / 'generated' / 'synthetic'
ensure_synthetic_dataset(
    DATASET_DIR,
    num_users=4000,
    num_campaigns=2500,
    num_interactions=120000,
    feature_count=12,
)

users = [UserProfile.model_validate(row) for row in read_jsonl(DATASET_DIR / 'users.jsonl')]
campaigns = [Campaign.model_validate(row) for row in read_jsonl(DATASET_DIR / 'campaigns.jsonl')]
interactions = pd.read_parquet(DATASET_DIR / 'interactions.parquet')
metadata = pd.read_json(DATASET_DIR / 'metadata.json', typ='series')

metadata

## Dataset Shape

The synthetic generator intentionally keeps the local data footprint small enough for a laptop demo while still preserving the structure of a real DSP retrieval path.

In [ ]:
pd.Series(
    {
        'users': len(users),
        'campaigns': len(campaigns),
        'interactions': len(interactions),
        'positive_labels': int(interactions['label'].sum()),
        'positive_rate': round(float(interactions['label'].mean()), 4),
    }
)

## User And Campaign Schemas

Users carry continuous interest scores and a small list of coarse segments.
Campaigns carry hard filters (`geo`, `device`, `required_segments`) plus weighted features for reranking.

In [ ]:
sample_user = users[0]
sample_campaign = campaigns[0]

display(pd.json_normalize(sample_user.model_dump()))
display(pd.json_normalize(sample_campaign.model_dump()))

## How Interests Become Segments

The service does not use every raw feature in Redis candidate generation.
Instead, the generator bins strong interests into segments such as `camping_high` or `travel_medium`.
Those segments are what land in Redis Sets and make `SINTER` selective.

In [ ]:
segment_frame = pd.DataFrame(
    [
        {
            'feature': feature,
            'interest': value,
            'derived_segment': segment_for(feature, value),
        }
        for feature, value in sorted(sample_user.interests.items(), key=lambda item: item[1], reverse=True)
    ]
)
segment_frame

## Label Generation

Synthetic labels come from a noisy latent truth model in `data.common`.
That truth model checks eligibility first, then combines user interests, campaign weights, bid, freshness, and deterministic noise.

This is important because it lets us ask two separate questions later:
- did candidate generation keep the relevant campaigns alive?
- once those campaigns survive, did the reranker put them near the top?

In [ ]:
eligible_campaign = next(campaign for campaign in campaigns if campaign_is_eligible(sample_user, campaign))

pd.Series(
    {
        'campaign_id': eligible_campaign.campaign_id,
        'eligible': campaign_is_eligible(sample_user, eligible_campaign),
        'truth_score': round(truth_score(sample_user, eligible_campaign), 4),
        'click_probability': round(click_probability(sample_user, eligible_campaign), 4),
        'label': click_label(sample_user, eligible_campaign),
    }
)

## Interaction Table

The interaction table is used for offline analysis, not for hot-path retrieval.
It gives us a reproducible click log with both positive and negative outcomes.

In [ ]:
display(interactions.head())
display(interactions.groupby(['eligible', 'label']).size().rename('rows').reset_index())